In [1]:
import solid
from solid.utils import *
import viewscad
import subprocess
import os
r = viewscad.Renderer(width=800, height=800)

In [7]:
#generate position list of wells using offset from center
wells_pos_from_center = lambda offset: [[-offset,offset],
                                        [offset, offset],
                                        [-offset, -offset],
                                        [offset, -offset]]

#place wells in four corners 
def four_corner(radius,height=None, positions = None, dxf = False):
    #if well pos not specified set
    #to equally spaced by half radius
    if positions is None:
        offset = radius+radius/2.0
        positions = wells_pos_from_center(offset)
    if dxf or (height is None):
        height = 0
    #make wells at positions with specified radius and height
    wells = []
    for position in positions:
        #make circle if making dxf
        if dxf:
            well_shape = solid.circle(r=radius, segments=64)
            well_shape = solid.translate([position[0],position[1]])(well_shape)
        #if stl make cylinder
        else:
            well_shape = solid.cylinder(r=radius, h=height, segments=64, center = True)
            well_shape = solid.translate([position[0],position[1],height/2.0])(well_shape)
        wells.append(well_shape)
    return  union()(*wells)

#generate channels 
def make_channels(length,width,height=0,num_chans=1,max_chans=None,spacing=None, dxf = False):
    if dxf or (height is None):
        height = 0
    #if channel gap not defined, is equal to
    if spacing is None: spacing = width
        
    #calculate number of channels if fitting max num within given width
    if not max_chans is None:
        import math
        num_chans = math.floor((max_chans-spacing)/(width+spacing))
        
    #get total width of channel section  
    total_width = (width*num_chans)+(spacing*(num_chans-1))
    
    print(length)
    #make template channel
    if dxf:
        channel_t = solid.square([length,width],center = True)
    else:
        channel_t = solid.cube([length,width,height], center = True)
    
    #if number channels is even, offset the first channel to center its gap
    if num_chans % 2 == 0: centering = -(width/2.0+spacing/2.0)
    else: centering = 0
    #move template channel up to align bottom at z = 0 
    #and offset center if necessary
    if dxf:
        channel_t = solid.translate([0,centering])(channel_t)
    else:
        channel_t = solid.translate([0,centering,height/2.0])(channel_t)
   
    #start making the channels
    channels = []
    for i in range(num_chans):
        #if i=0 there is no translation
        #if i=1, we make the first channel adjacent to second
        direction = i
        
        #if i>=1, we add channels alternating left and right
        if i >= 1:
            direction = -(i/2.0)
            if i % 2 == 1: direction = (-direction)+0.5
                
        #make channel at right position and add it to channel list
        if dxf:
            channel = solid.translate([0,direction*(width+spacing)])(channel_t)
        else:
            channel = solid.translate([0,direction*(width+spacing),0])(channel_t)
        channels.append(channel)
    
    #group all channels as one object
    channels = union()(*channels)
    #compute the x,y,z dimensions in order to pass as arguments 
    #for construction of other parts of device (such as the chambers)
    measurements = {'x':(length/2.0,-length/2.0),
                    'y':(total_width/2.0,-total_width/2.0)}
    if not dxf:
        measurements['z'] = (height,0)
    return channels,measurements

def make_chambers(msrs,height=None,extra=0,len_until=None, dxf = False):
    #computes x,y,z length
    total = lambda x : abs(x[0]) + abs(x[1])
    
    #if specific height if specified
    #otherwise use same height as channels
    if dxf:
        pass
    elif not height is None:
        msrs['z'] = (height,0)
    else:
        msrs['z'] = (0,0)
        
    #make copy of measurements from channels to modify chamber dimensions 
    #and add length to chambers as specified
    chamber_dims = msrs.copy()
    #calculate length of chamber to reach specified offset
    if not len_until is None:
        chamber_len = len_until-(msrs['x'][0])
    #or add specified length to channel length (if not specified, adds 0)
    else:
        chamber_len = msrs['x'][0]+extra
    #set the chamber length based 
    chamber_dims['x'] = (chamber_len/2.0,-(chamber_len/2.0))
    #compute final dimensions of chamber
    chamber_dims = [total(size) for size in chamber_dims.values()]
        
    #calculate chamber translation offset to place adjacent to channels
    chamber_trslt = msrs['x'][0]+(chamber_len/2.0)
    #compute chamber translation coordinates to align with channels
    #x = 1 moves chamber above channels
    #x = -1 moves chamber below channels
    if dxf:
        trslt = lambda x : [x*(chamber_trslt),0]
    else:
        trslt = lambda x : [x*(chamber_trslt),0,msrs['z'][0]/2.0]
    
    #function to make chamber at top or at bottom 
    if dxf:
        move = lambda x : solid.translate(trslt(x))(solid.square(chamber_dims, center = True) )
    else:
        move = lambda x : solid.translate(trslt(x))(solid.cube(chamber_dims,center = True))
    #make top and bottom chambers and return them as grouped
    top = move(1)
    bottom = move(-1)
    return solid.union()(top,bottom)

#create alignment features 
def alignment_features(unit,dims,grid_size, mask_size = None, alignment = "hollow", units_from_center = (2.5,2.5)):
    height = grid_size[0]*dims[0]
    length = grid_size[1]*dims[1]
    if mask_size is None:
        corner_len = (dims[0] + dims[1]) /2 / 8
    corner = lambda thickness_div : solid.union()( solid.square([corner_len,corner_len/thickness_div],center=False), solid.square([corner_len/thickness_div,corner_len],center=False) )
    
    def make_full(thickness_div):
        thickness = corner_len/thickness_div
        tr = solid.translate([0, 0, 0 ])(solid.rotate(180)(corner(thickness_div)))
        tr = solid.translate([thickness/2,thickness/2,0])(tr)
        bl = solid.translate([0, 0, 0 ])(solid.rotate(0)(corner(thickness_div)))
        bl = solid.translate([-thickness/2,-thickness/2,0])(bl)
        return solid.union()(tr,bl)
    
    def make_hollow(thickness_div):
        thickness = corner_len/(thickness_div/2)
        inner = make_full(thickness_div)
        outer = make_full(thickness_div/2)
        hollow = outer - inner 
        return hollow
    
    center = (height /2 , length/2)
    mask_pos = lambda x,y : (center[0] + x*(float(units_from_center[0])*dims[0]), center[1] + y*(float(units_from_center[1])*dims[1]))
    positions = []
    positions.append(mask_pos(1,1))
    positions.append(mask_pos(-1,1))
    positions.append(mask_pos(1,-1))
    positions.append(mask_pos(-1,-1))
    make_mask = lambda mtype : make_hollow(8) if mtype == "hollow" else make_full(8)
    masks = [ solid.translate([*position,0])(make_mask(alignment)) for position in positions]

    return unit - solid.union()(*masks)

def create_outline(thickness, array, dims, grid_size):
    height = grid_size[0]*dims[0] + thickness *2
    length = grid_size[1]*dims[1] + thickness *2
    outer = solid.translate([-thickness, -thickness,0])(solid.square([height, length]))
    return outer-array
    
        
#build a row x col grid of containment units
def make_unit_array(unit,dims,grid_size, dxf = False, alignment = None, mask_size = None,
                    units_from_center = None, outline_thickness = 0.050):
    units = []
    for col in range(grid_size[1]):
        for row in range(grid_size[0]):
            if dxf:
                units.append(solid.translate([row*dims[0],col*dims[1]])(unit))
            else:
                units.append(solid.translate([row*dims[0],col*dims[1],dims[2]/2.0])(unit))
    array = solid.translate([dims[0]/2,dims[1]/2])(solid.union()(*units) )
    if (not alignment is None) and dxf:
        array = alignment_features(array,dims, grid_size, alignment = alignment, mask_size = None, units_from_center = units_from_center )
    return create_outline(outline_thickness, array, dims, grid_size)
    
    

    

In [19]:
"""
Premise: 
Make array of two-compartment containment units to seperate neuronal
cell bodies from axons. Micro-channels will connect both chambers 
allowing axons to cross and be isolated.
Wells must be placed to allow a 12 channel pipette to enter the
same position well for all adjacent containment units.
For example, all pipette tips will be enter the top left well of all adjacent wells

I mostly used this article and the cited papers as a reference for dimensions:
https://europepmc.org/article/PMC/5831486
For comparison to what we have been using in the lab, you can take a look at this page:
https://anandadevices.com/shop/migration-device-medium-sample/
"""

"""
Wells:

Well Positions:
The distance between two wells is 9mm, therefore all wells must be 4.5mm apart
to allow the tips to enter the same well of each containment unit.
If wells must be 4.5mm apart, they are 2.25mm away from the center in X and Y
(4.5mm/2 = 2.25mm)
"""
wells_pos = wells_pos_from_center(2.25)

"""
Well Radius:
I found biopsy punchers available in 2mm and 3mm. 2mm is very small and going over
3mm leads to little space between wells, so I thought 3mm would be a good compromise 
"""
well_rad = 1.5 #3mm/2

"""
Well height:
I don't think this is too critical. I'm not too familiar on how the thickness of the
device can affect fabrication. This parameter can be freely changed without affecting
the functional design.
"""
well_height = 2
#well_height = 0



"""
Channels:
The main goal is to make channels big enough for axons to pass but small enough 
to block cell migration. There shouldn't really be cell migration or proliferation
since these will be used with cortical neuron cultures. However the cells could 
enter and block the channels during their seeding.

Width:
I have seen axon microchannels designed with widths from 5um to 20uM. 
I am not really considering widths under 10 um due to the drastic price
increase.  
"""
chan_w = 0.015

"""
Length:
The ananda devices we are using are advertised to have channels at least 1mm long.
We don't really need the channels to be very long, but I set them to be 1mm for 
consistency.
"""
chan_l = 1
"""
Height:
Smaller is better like with the width. 
"""
chan_h = 0.010 
#chan_h = 0
"""
Channel Gap:
I've read that this plays an important role in prevent the channels from collapsing. 
I guestimated that at least double the width of the channel should suffice
"""
chan_gap = 0.030
"""
Number of Channels:
This is set to fill up the width defined by the distance between the center 
of two adjacent wells. You can change all the parameters above and the numbers of wells will 
adapt. You can also set the number of wells manually by using the num_well keyword parameter
and removing the max_chans argument that is currently being passed to the make_channels
function below.
"""
max_chans = 2.25*2 #distance between centers of two adjacent wells

"""
Chambers:

Height:
I found that 100um was used in a few devices from the review paper 
so I chose that. I don't think this is parameter too important.
"""
chamber_height = 0.100
#chamber_height = 0

"""
Array of containment units size:
I don't know what the max dimensions, so I made a function to build a gride of units 
of arbitrary rows by columns size. Simply edit the values of the variables below.
"""
rows = 1
columns = 1

"""
This part of the cell calls functions that use the parameters defined above to 
build a single containment unit. I've set this up so that modifying the design
can be done by changing the values above without having to change any of the 
function calls below. If you'd like to make some more specfic changes you can
take a look at the helper function definitions in the cell above. I've left comments 
to try and make the code clear.
"""

#construct pieces of containment unit
wells = four_corner(well_rad,well_height,positions = wells_pos)
channels, msrs = make_channels(chan_l,chan_w,chan_h,spacing = chan_gap, max_chans=2.25*2)
chambers = make_chambers(msrs,height=chamber_height,len_until=2.25)

#add together and render
negative=union()(wells,channels,chambers)

#make casing for one containment unit
casing_x = 9.0 # the 9 is from the distance between each pipette tip in a 96 well
casing_y = casing_x
casing_z = well_height
casing = solid.translate([0,0,casing_z/2.0])(
    solid.cube([casing_x,casing_y,casing_z],center=True)
)

#extrude wells, chambers and channels from casing 
unit = casing-negative

#flip containment unit upside down for visualization
unit = solid.translate([0,0,-casing_z/2])(rot_z_to_down(unit))

#build array of units
unit_array = make_unit_array(unit,[casing_x,casing_y,casing_z],[rows,columns])

r.render(unit_array, outfile='1x1_units.stl')

[-2.25, 2.25]
[2.25, 2.25]
[-2.25, -2.25]
[2.25, -2.25]


In [4]:
r.render(unit,outfile='1x1_unit.stl')

In [8]:

params = { 'wells_pos': wells_pos_from_center(2.25),
           'well_rad' : 1.5 ,
           'well_height' : 2,
           'chan_w'  : 0.01,
           'chan_l'  : 0.8,
           'chan_h'  : 0.010 ,
           'chan_gap'  : 0.040,
           'max_chans'  : 2.25*2, 
           'chamber_height'  : 0.100,
           'columns'  : 1,
           'rows'  : 1,
           'add_channels' : True,
           'add_wells' : True,
           'add_chambers' : True,
           'save_path' : "./",
           'render_stl' : False,
           'dxf' : True
         }

def make_device(wells_pos = wells_pos_from_center(2.25), well_rad = 1.5 ,
                well_height = 2, chan_w = 0.01, chan_l = 1, chan_h = 0.010 ,
                chan_gap = 0.040, max_chans = 2.25*2, chamber_height  = 0.100,
                columns = 1, rows = 1, add_channels = True, add_wells = True, 
                add_chambers = True, save_path = "./", alignment = None, units_from_center = 2,
                mask_size = None, render_stl = False, outline_thickness = 0.050, dxf = True):
    
    wells = four_corner(well_rad,well_height, dxf = dxf, positions = wells_pos)
    file_name = str(rows)+'x'+str(columns)+'_units_'

    channels, msrs = make_channels(chan_l,chan_w,chan_h,dxf=dxf, spacing = chan_gap, max_chans=max_chans)
    chambers = make_chambers(msrs,dxf = dxf, height=chamber_height,len_until=2.25)
    #add together and render
    to_add = []
    if add_channels:
        to_add.append(channels)
        file_name = "chans_" + file_name
    if add_wells:
        to_add.append(wells)
        file_name = "wells_" + file_name
    if add_chambers:
        to_add.append(chambers)
        file_name = "chambers_" + file_name
    negative=union()(*to_add)
    
    #make casing for one containment unit
    casing_x = 9.0 # the 9 is from the distance between each pipette tip in a 96 well
    casing_y = casing_x
    casing_z = 0
    if dxf:
        casing = solid.translate([0,0])(
            solid.square([casing_x,casing_y],center=True)
        )
    else:
        casing_z = well_height
        casing = solid.translate([0,0,casing_z/2.0])(
            solid.cube([casing_x,casing_y,casing_z],center=True)
        )
    
    #extrude wells, chambers and channels from casing 
    unit = casing-negative
    
    if not dxf:
        #flip containment unit upside down for visualization
        unit = solid.translate([0,0,-casing_z/2.0])(rot_z_to_down(unit))
    
    #build array of units
    grid_size = [rows,columns]
    dims = [casing_x, casing_y, casing_z]
    unit_array = make_unit_array(unit,dims,grid_size, dxf=dxf, alignment = alignment, units_from_center = units_from_center)
    ret_fname = os.path.abspath(save_path+file_name+".scad")
    solid.scad_render_to_file(unit_array,ret_fname)
  #  if render_stl: 
  #      ret_fname = os.path.abspath(save_path+file_name+".stl")
  #      r.render(unit_array, outfile=ret_fname)
    return ret_fname
    
def to_dxf(scad_path):
    dxf_path = scad_path.replace(".scad",".dxf")
    subprocess.call(["openscad", "-o", dxf_path, scad_path])
    import ezdxf
    temp_dxf = ezdxf.readfile(dxf_path)
    temp_dxf.saveas(dxf_path)


In [41]:
#1x1
params['rows'] = 1
params['columns'] = 1 
params['save_path'] = "./designs/"
params['dxf'] = True
params['add_channels'] = True
params['add_wells'] = True
params['add_chambers'] = True
full = make_device(**params)
to_dxf(full)
print('1x1 full complete')

params['add_channels'] = False
no_chans = make_device(**params) 
to_dxf(no_chans)
print('1x1 no_chans complete')

#12x8
params['rows'] = 12
params['columns'] = 8 
params['add_channels'] = True
full = make_device(**params)
to_dxf(full)
print('12x8 full complete')

params['add_channels'] = False
no_chans = make_device(**params) 
to_dxf(no_chans)
print('12x8 no_chans complete')

1x1 full complete
1x1 no_chans complete
12x8 full complete
12x8 no_chans complete


In [38]:
#dxf
params['rows'] = 4
params['columns'] = 4 
params['save_path'] = "./designs/"
params['add_channels'] = True
params['dxf'] = True
full = make_device(**params)
#to_dxf(full)
print('1x1 full complete')

1x1 full complete


In [ ]:
#stl
params['rows'] = 1
params['columns'] = 1 
params['save_path'] = "./designs/"
params['add_channels'] = True
params['dxf'] = False
full = make_device(**params)
#to_dxf(full)
print('1x1 full complete')

In [11]:
params = { 'wells_pos': wells_pos_from_center(2.25),
           'well_rad' : 1.5 ,
           'well_height' : 2,
           'chan_w'  : 0.01,
           'chan_l'  : 0.8,
           'chan_h'  : 0.010 ,
           'chan_gap'  : 0.040,
           'max_chans'  : 2.25*2, 
           'chamber_height'  : 0.100,
           'columns'  : 1,
           'rows'  : 1,
           'add_channels' : True,
           'add_wells' : True,
           'add_chambers' : True,
           'save_path' : "./",
           'render_stl' : False,
           'dxf' : True
         }

#8x12 layer 1
params['rows'] = 8
params['columns'] = 12
params['save_path'] = "./designs/"
params['chan_l'] = 1.4
params['dxf'] = True
params['add_channels'] = True
params['add_wells'] = False
params['add_chambers'] = False
params['alignment'] = "full"
params['units_from_center'] = (3,1.5)
params['mask_size'] = 500
full = make_device(**params)
to_dxf(full)
print('8x12 full complete')

#8x12 layer 2
params['rows'] = 8
params['columns'] = 12 
params['save_path'] = "./designs/"
params['dxf'] = True
params['add_channels'] = False
params['add_wells'] = True
params['add_chambers'] = True
params['chan_l'] = 0.8
params['alignment'] = "hollow"
params['units_from_center'] = (3,1.5)
params['mask_size'] = 500
full = make_device(**params)
to_dxf(full)
print('8x12 full complete')


1.4
8x12 full complete
0.8
8x12 full complete
